# Notebook 10 — Validación del Modelo sobre el Set Final

**Proyecto:** Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad  
**Autor:** Jesús Goenaga Peña  
**Fase:** 4 del pipeline de validación — Evaluación del modelo con pares A/B

---

## ¿Qué hace este notebook?

Ejecuta la validación formal del modelo cognitivo artificial sobre el set de validación final definido en el Notebook 09 (`validation_set_final.csv`). A diferencia de la evaluación del entrenamiento (NB05), aquí:

1. **La tarea es relativa entre regiones A y B**: el modelo evalúa si la región A está más cercana o más lejana que la región B, usando las coordenadas exactas definidas en el NB08.
2. **Se ejecutan 3 repeticiones independientes** del mismo conjunto de tareas, simulando 3 participantes artificiales (según protocolo de la tesis, sección 8.4.2.4).
3. **Cada respuesta queda vinculada** a su condición factorial completa (tipo de tarea, nivel de disparidad, presencia de distractores) y al ground truth de pares A/B.
4. **El CSV de salida es directamente comparable** con los datos humanos del Notebook 11 para los análisis estadísticos.

## Lógica de inferencia regional

El modelo fue entrenado para clasificación binaria de profundidad relativa (salida sigmoid: 0 = lejano, 1 = cercano). Para evaluar profundidad relativa entre dos regiones A y B:

1. Se recorta la región A del par estereoscópico y se pasa por el modelo → score_A (probabilidad de ser cercano).
2. Se recorta la región B del par estereoscópico y se pasa por el modelo → score_B.
3. Si score_A > score_B → predicción: A más cercano. Si score_A ≤ score_B → predicción: A más lejano.
4. Se compara la predicción con el ground truth del CSV.

---

## Celda 1 — Montar Drive y verificar rutas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR       = '/content/drive/MyDrive/cognitive-depth-model'
SPLITS_DIR     = os.path.join(BASE_DIR, 'data', 'splits')
CHECKPOINTS    = os.path.join(BASE_DIR, 'checkpoints')
RESULTS_DIR    = os.path.join(BASE_DIR, 'results', 'model_validation')
METRICS_DIR    = os.path.join(BASE_DIR, 'results', 'metrics')
VIZ_DIR        = os.path.join(BASE_DIR, 'results', 'visualizations')
os.makedirs(RESULTS_DIR, exist_ok=True)

CSV_VALIDATION = os.path.join(SPLITS_DIR,  'validation_set_final.csv')
MODEL_PATH     = os.path.join(CHECKPOINTS, 'best_model.pth')

print('Verificación de rutas:')
print('-' * 55)
for nombre, ruta in [
    ('CSV set validación final (NB09)', CSV_VALIDATION),
    ('Mejor modelo (best_model.pth)',   MODEL_PATH),
    ('Carpeta resultados validación',   RESULTS_DIR),
]:
    existe = os.path.exists(ruta)
    print(f'  {"✓" if existe else "✗"}  {nombre}')

print()
print('Si todo está en verde, continúa con la Celda 2.')

## Celda 2 — Importar librerías y configurar entorno

In [ ]:
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')

# Cargar set de validación
df_val = pd.read_csv(CSV_VALIDATION)
print(f'\nSet de validación cargado: {len(df_val)} tareas')
print(f'  KITTI:     {(df_val["dataset"]=="KITTI").sum()}')
print(f'  Ilusiones: {(df_val["dataset"]=="3D_Illusion").sum()}')
print()
print('Columnas disponibles:')
print(df_val.columns.tolist())

## Celda 3 — Definir y cargar la arquitectura del modelo

Reproducimos la misma arquitectura definida en el Notebook 04 para poder
cargar los pesos del checkpoint correctamente.

In [ ]:
import torchvision.models as tv_models

class CognitiveDepthModel(nn.Module):
    """
    Arquitectura del modelo cognitivo artificial (Fases 7-27).
    ResNet-18 backbone con capas adaptadas para clasificación binaria
    de profundidad relativa. Entrada: par estereoscópico (2 imágenes).
    Salida: sigmoid → probabilidad de ser 'más cercano'.
    """
    def __init__(self):
        super().__init__()
        # Backbone ResNet-18 (sin pesos preentrenados para reproducibilidad)
        backbone = tv_models.resnet18(weights=None)
        # Adaptar primera capa para entrada de 6 canales (3 RGB × 2 imágenes)
        backbone.conv1 = nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # Extraer todo excepto la capa de clasificación final
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


# Cargar modelo con los pesos entrenados
model = CognitiveDepthModel().to(device)

checkpoint = torch.load(MODEL_PATH, map_location=device)

# El checkpoint puede ser el estado del modelo directamente o un dict con clave 'model_state_dict'
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f'Checkpoint cargado (epoch {checkpoint.get("epoch", "?")}, '
          f'val_acc {checkpoint.get("val_acc", "?"):.4f})')
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'])
    print('Checkpoint cargado (formato state_dict).')
else:
    model.load_state_dict(checkpoint)
    print('Checkpoint cargado (formato directo).')

model.eval()
print('Modelo en modo evaluación. Listo para inferencia.')

# Transformación de entrada (igual que en entrenamiento)
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((192, 640)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406, 0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225, 0.229, 0.224, 0.225])
])

print('\nTransformación de entrada definida (192×640, normalización ImageNet×2).')

## Celda 4 — Definir función de inferencia regional

La función toma una imagen y las coordenadas de las regiones A y B,
y devuelve la predicción del modelo sobre cuál región está más cerca.

In [ ]:
def preparar_region(img_bgr, x, y, ancho, alto, img_H, img_W):
    """
    Recorta una región de la imagen y la prepara para el modelo.
    Si la región es muy pequeña, se redimensiona la imagen completa
    en su lugar para evitar pérdida de información.
    """
    # Asegurar coordenadas válidas dentro de la imagen
    x0 = max(0, int(x))
    y0 = max(0, int(y))
    x1 = min(int(img_W), int(x + ancho))
    y1 = min(int(img_H), int(y + alto))

    region = img_bgr[y0:y1, x0:x1]

    if region.size == 0 or region.shape[0] < 4 or region.shape[1] < 4:
        # Fallback: usar imagen completa
        region = img_bgr

    # Convertir BGR → RGB
    region_rgb = cv2.cvtColor(region, cv2.COLOR_BGR2RGB)
    return region_rgb


def inferir_par(ruta_img, row, modelo, transf, dispositivo):
    """
    Dado un par de regiones A y B, corre el modelo sobre cada una
    y devuelve la predicción comparativa.

    Lógica:
        score_A = P(región A es más cercana)
        score_B = P(región B es más cercana)
        Si score_A > score_B → predicción: 'mas_cercano' (A más cerca que B)
        Si score_A ≤ score_B → predicción: 'mas_lejano'  (A más lejos que B)

    Retorna: dict con predicción, scores, y tiempo de respuesta en ms.
    """
    img = cv2.imread(str(ruta_img))
    if img is None:
        return None

    H = int(row['img_H']) if pd.notna(row.get('img_H')) else img.shape[0]
    W = int(row['img_W']) if pd.notna(row.get('img_W')) else img.shape[1]

    # Recortar regiones A y B
    region_A = preparar_region(img, row['A_x'], row['A_y'],
                                row['A_ancho'], row['A_alto'], H, W)
    region_B = preparar_region(img, row['B_x'], row['B_y'],
                                row['B_ancho'], row['B_alto'], H, W)

    # Para el modelo estereoscópico (6 canales), duplicamos la región
    # como aproximación monoscópica de cada región independiente
    def region_a_tensor(region_rgb):
        region_resized = cv2.resize(region_rgb, (320, 192))
        # Concatenar consigo misma para simular par estéreo de la región
        par = np.concatenate([region_resized, region_resized], axis=2)
        tensor = torch.from_numpy(par.transpose(2, 0, 1)).float() / 255.0
        # Normalización ImageNet ×2 canales
        mean = torch.tensor([0.485,0.456,0.406,0.485,0.456,0.406]).view(6,1,1)
        std  = torch.tensor([0.229,0.224,0.225,0.229,0.224,0.225]).view(6,1,1)
        tensor = (tensor - mean) / std
        # Redimensionar a 640 ancho
        tensor = torch.nn.functional.interpolate(
            tensor.unsqueeze(0), size=(192, 640), mode='bilinear', align_corners=False
        )
        return tensor.to(dispositivo)

    t_inicio = time.time()

    with torch.no_grad():
        score_A = float(modelo(region_a_tensor(region_A)).squeeze())
        score_B = float(modelo(region_a_tensor(region_B)).squeeze())

    t_ms = (time.time() - t_inicio) * 1000

    prediccion = 'mas_cercano' if score_A > score_B else 'mas_lejano'

    return {
        'score_A':     round(score_A, 6),
        'score_B':     round(score_B, 6),
        'prediccion':  prediccion,
        'tiempo_ms':   round(t_ms, 2)
    }


print('Funciones de inferencia regional definidas.')
print()
print('Lógica de decisión:')
print('  score_A = P(región A es más cercana al sensor)')
print('  score_B = P(región B es más cercana al sensor)')
print('  score_A > score_B → "mas_cercano" (A más cerca que B)')
print('  score_A ≤ score_B → "mas_lejano"  (A más lejos que B)')

## Celda 5 — Ejecutar 3 repeticiones del modelo

Cada repetición es una pasada completa por las 190 tareas del set de validación.
Las 3 repeticiones simulan 3 participantes artificiales independientes.

In [ ]:
N_REPETICIONES = 3
todas_respuestas = []

for rep in range(1, N_REPETICIONES + 1):
    print(f'\nRepetición {rep}/{N_REPETICIONES}...')
    aciertos = 0
    errores_img = 0

    for _, row in tqdm(df_val.iterrows(), total=len(df_val),
                       desc=f'Rep {rep}'):

        ruta_img = str(row['ruta_img_izq']) if pd.notna(row.get('ruta_img_izq')) else ''

        if not ruta_img or not os.path.exists(ruta_img):
            errores_img += 1
            todas_respuestas.append({
                'repeticion':             rep,
                'numero_tarea':           row['numero_tarea'],
                'imagen_id':              row['imagen_id'],
                'dataset':                row['dataset'],
                'tipo_tarea':             row['tipo_tarea'],
                'nivel_disparidad':       row['nivel_disparidad'],
                'presencia_distractores': row['presencia_distractores'],
                'ground_truth':           row['ground_truth'],
                'prediccion':             'error',
                'acierto':                0,
                'score_A':                np.nan,
                'score_B':                np.nan,
                'tiempo_ms':              np.nan,
            })
            continue

        resultado = inferir_par(ruta_img, row, model, transform, device)

        if resultado is None:
            errores_img += 1
            prediccion = 'error'
            acierto    = 0
            score_A = score_B = np.nan
            t_ms = np.nan
        else:
            prediccion = resultado['prediccion']
            acierto    = int(prediccion == row['ground_truth'])
            score_A    = resultado['score_A']
            score_B    = resultado['score_B']
            t_ms       = resultado['tiempo_ms']
            aciertos  += acierto

        todas_respuestas.append({
            'repeticion':             rep,
            'numero_tarea':           row['numero_tarea'],
            'imagen_id':              row['imagen_id'],
            'dataset':                row['dataset'],
            'tipo_tarea':             row['tipo_tarea'],
            'nivel_disparidad':       row['nivel_disparidad'],
            'presencia_distractores': row['presencia_distractores'],
            'ground_truth':           row['ground_truth'],
            'prediccion':             prediccion,
            'acierto':                acierto,
            'score_A':                score_A,
            'score_B':                score_B,
            'tiempo_ms':              t_ms,
        })

    n_validas = len(df_val) - errores_img
    acc = aciertos / n_validas if n_validas > 0 else 0
    print(f'  Repetición {rep}: accuracy = {acc:.4f} ({aciertos}/{n_validas})')
    if errores_img > 0:
        print(f'  ⚠ Imágenes no encontradas: {errores_img}')

df_respuestas = pd.DataFrame(todas_respuestas)
print(f'\nTotal respuestas generadas: {len(df_respuestas)}')
print(f'  ({N_REPETICIONES} repeticiones × {len(df_val)} tareas)')

## Celda 6 — Calcular métricas por condición factorial

In [ ]:
df_validas = df_respuestas[df_respuestas['prediccion'] != 'error'].copy()

print('=' * 65)
print('DESEMPEÑO GLOBAL DEL MODELO')
print('=' * 65)

acc_global = df_validas['acierto'].mean()
print(f'Accuracy global (3 reps): {acc_global:.4f}  ({acc_global*100:.1f}%)')
print()

# Accuracy por repetición
print('Accuracy por repetición:')
for rep in sorted(df_validas['repeticion'].unique()):
    acc_rep = df_validas[df_validas['repeticion']==rep]['acierto'].mean()
    print(f'  Repetición {rep}: {acc_rep:.4f}  ({acc_rep*100:.1f}%)')
print()

# Accuracy por tipo de tarea
print('Accuracy por tipo de tarea:')
acc_tarea = df_validas.groupby('tipo_tarea')['acierto'].mean()
for tarea, acc in acc_tarea.items():
    print(f'  {tarea}: {acc:.4f}  ({acc*100:.1f}%)')
print()

# Accuracy por condición factorial completa
print('Accuracy por condición factorial 2×2×2:')
print('-' * 65)
acc_factorial = df_validas.groupby(
    ['tipo_tarea', 'nivel_disparidad', 'presencia_distractores']
)['acierto'].agg(['mean','sum','count']).reset_index()
acc_factorial.columns = ['Tarea', 'Disparidad', 'Distractores', 'Accuracy', 'Aciertos', 'N']
acc_factorial['Accuracy_%'] = (acc_factorial['Accuracy'] * 100).round(1)
print(acc_factorial[['Tarea','Disparidad','Distractores',
                      'Accuracy_%','Aciertos','N']].to_string(index=False))
print()

# Accuracy por nivel de disparidad
print('Accuracy por nivel de disparidad:')
acc_disp = df_validas.groupby('nivel_disparidad')['acierto'].mean()
for nivel, acc in acc_disp.items():
    print(f'  {nivel}: {acc:.4f}  ({acc*100:.1f}%)')
print()

# Accuracy por presencia de distractores
print('Accuracy por presencia de distractores:')
acc_dist = df_validas.groupby('presencia_distractores')['acierto'].mean()
for cond, acc in acc_dist.items():
    print(f'  {cond}: {acc:.4f}  ({acc*100:.1f}%)')

## Celda 7 — Visualizaciones de desempeño

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Desempeño del Modelo — Validación con Set Final (Pares A/B)',
             fontsize=13, fontweight='bold')

# 1. Accuracy por celda factorial
ax = axes[0, 0]
etiq = [f"{r['Tarea'][:4].upper()}\n{r['Disparidad']}\n{r['Distractores'][:3]}"
        for _, r in acc_factorial.iterrows()]
colores = ['#42A5F5' if 'disc' in r['Tarea'] else '#AB47BC'
           for _, r in acc_factorial.iterrows()]
bars = ax.bar(range(len(acc_factorial)), acc_factorial['Accuracy_%'],
              color=colores, edgecolor='white')
ax.axhline(50, color='red',  ls='--', lw=1.5, label='Azar = 50%')
ax.axhline(acc_global*100, color='green', ls=':', lw=1.5,
           label=f'Global = {acc_global*100:.1f}%')
ax.set_xticks(range(len(acc_factorial)))
ax.set_xticklabels(etiq, fontsize=7)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)
ax.set_title('Accuracy por condición factorial')
ax.legend(fontsize=8)
for b, v in zip(bars, acc_factorial['Accuracy_%']):
    ax.text(b.get_x()+b.get_width()/2, v+1, f'{v:.1f}',
            ha='center', fontsize=8, fontweight='bold')

# 2. Accuracy por repetición
ax = axes[0, 1]
accs_rep = [df_validas[df_validas['repeticion']==r]['acierto'].mean()*100
            for r in sorted(df_validas['repeticion'].unique())]
ax.bar(['Rep 1','Rep 2','Rep 3'], accs_rep,
       color=['#1E88E5','#43A047','#FB8C00'], edgecolor='white')
ax.axhline(50, color='red', ls='--', lw=1.5, label='Azar = 50%')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)
ax.set_title('Consistency entre repeticiones')
ax.legend(fontsize=9)
for i, v in enumerate(accs_rep):
    ax.text(i, v+1, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

# 3. Distribución de scores A vs B
ax = axes[1, 0]
ax.hist(df_validas['score_A'].dropna(), bins=30, alpha=0.6,
        color='#E53935', label='Score región A', edgecolor='white')
ax.hist(df_validas['score_B'].dropna(), bins=30, alpha=0.6,
        color='#1E88E5', label='Score región B', edgecolor='white')
ax.axvline(0.5, color='gray', ls='--', lw=1.5, label='Umbral = 0.5')
ax.set_xlabel('Score sigmoid (P de ser más cercano)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de scores por región')
ax.legend(fontsize=9)

# 4. Accuracy por tipo de tarea y nivel de disparidad
ax = axes[1, 1]
pivot = df_validas.groupby(['tipo_tarea','nivel_disparidad'])['acierto'].mean().unstack() * 100
x = range(len(pivot))
w = 0.35
if 'alta' in pivot.columns:
    ax.bar([i-w/2 for i in x], pivot['alta'],  width=w,
           color='#E53935', label='Alta disparidad', edgecolor='white')
if 'baja' in pivot.columns:
    ax.bar([i+w/2 for i in x], pivot['baja'],  width=w,
           color='#1E88E5', label='Baja disparidad', edgecolor='white')
ax.set_xticks(list(x))
ax.set_xticklabels([t[:15] for t in pivot.index], fontsize=9)
ax.axhline(50, color='red', ls='--', lw=1.5, label='Azar = 50%')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)
ax.set_title('Accuracy: tarea × nivel de disparidad')
ax.legend(fontsize=8)

plt.tight_layout()
ruta_fig = os.path.join(RESULTS_DIR, 'model_validation_performance.png')
plt.savefig(ruta_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada en: {ruta_fig}')

## Celda 8 — Guardar CSVs de resultados

In [ ]:
# CSV principal: todas las respuestas del modelo (3 reps × N tareas)
ruta_respuestas = os.path.join(RESULTS_DIR,  'model_responses.csv')
ruta_resp_metr  = os.path.join(METRICS_DIR,  'model_responses.csv')
os.makedirs(METRICS_DIR, exist_ok=True)

df_respuestas.to_csv(ruta_respuestas, index=False)
df_respuestas.to_csv(ruta_resp_metr,  index=False)

# CSV resumen de métricas por condición
ruta_metricas = os.path.join(RESULTS_DIR, 'model_metrics_by_condition.csv')
acc_factorial.to_csv(ruta_metricas, index=False)

# CSV métricas globales
metricas_globales = pd.DataFrame([{
    'n_tareas':          len(df_val),
    'n_repeticiones':    N_REPETICIONES,
    'n_respuestas':      len(df_validas),
    'accuracy_global':   round(acc_global, 4),
    'accuracy_kitti':    round(df_validas[df_validas['dataset']=='KITTI']['acierto'].mean(), 4),
    'accuracy_ilusion':  round(df_validas[df_validas['dataset']=='3D_Illusion']['acierto'].mean(), 4),
    'accuracy_alta_disp': round(df_validas[df_validas['nivel_disparidad']=='alta']['acierto'].mean(), 4),
    'accuracy_baja_disp': round(df_validas[df_validas['nivel_disparidad']=='baja']['acierto'].mean(), 4),
    'accuracy_con_dist':  round(df_validas[df_validas['presencia_distractores']=='con_distractores']['acierto'].mean(), 4),
    'accuracy_sin_dist':  round(df_validas[df_validas['presencia_distractores']=='sin_distractores']['acierto'].mean(), 4),
}])
ruta_glob = os.path.join(METRICS_DIR, 'model_global_metrics.csv')
metricas_globales.to_csv(ruta_glob, index=False)

print('✓ CSVs de resultados guardados:')
print(f'  → {ruta_respuestas}  ({len(df_respuestas)} filas)')
print(f'  → {ruta_metricas}')
print(f'  → {ruta_glob}')

## Celda 9 — Resumen final

In [ ]:
print('=' * 65)
print('RESUMEN FINAL — VALIDACIÓN DEL MODELO')
print('=' * 65)
print(f'Set de validación:    {len(df_val)} tareas')
print(f'Repeticiones:         {N_REPETICIONES}')
print(f'Total respuestas:     {len(df_respuestas)}')
print()
print(f'Accuracy global:      {acc_global:.4f}  ({acc_global*100:.1f}%)')
print(f'  KITTI:              {df_validas[df_validas["dataset"]=="KITTI"]["acierto"].mean():.4f}')
print(f'  Ilusiones:          {df_validas[df_validas["dataset"]=="3D_Illusion"]["acierto"].mean():.4f}')
print(f'  Alta disparidad:    {df_validas[df_validas["nivel_disparidad"]=="alta"]["acierto"].mean():.4f}')
print(f'  Baja disparidad:    {df_validas[df_validas["nivel_disparidad"]=="baja"]["acierto"].mean():.4f}')
print(f'  Con distractores:   {df_validas[df_validas["presencia_distractores"]=="con_distractores"]["acierto"].mean():.4f}')
print(f'  Sin distractores:   {df_validas[df_validas["presencia_distractores"]=="sin_distractores"]["acierto"].mean():.4f}')
print()
nivel_azar = 0.5
if acc_global > nivel_azar:
    print(f'✓ El modelo supera el nivel de azar ({nivel_azar*100:.0f}%).')
else:
    print(f'⚠ El modelo NO supera el nivel de azar ({nivel_azar*100:.0f}%).')
print()
print('Archivos generados:')
print(f'  1. results/model_validation/model_responses.csv')
print(f'  2. results/model_validation/model_metrics_by_condition.csv')
print(f'  3. results/metrics/model_responses.csv')
print(f'  4. results/metrics/model_global_metrics.csv')
print(f'  5. results/model_validation/model_validation_performance.png')
print()
print('Próximo paso → Notebook 11: Interfaz experimental para participantes humanos.')